# 260422 LangChain 내장 도구 (웹 검색)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w7_tool_calling/llm_260422_builtin_tools.ipynb)

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-openai openai

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 강의 메모: LangChain 내장 도구

어제는 `@tool` 데코레이터로 **커스텀 툴**을 만들었다면, 오늘은 LangChain이 기본 제공하는 **내장 툴**(Tavily / Wikipedia / DuckDuckGo / Python REPL 등)을 사용한다.

### 비유: MCP가 왜 나왔나
서비스 m개와 툴 n개를 직접 연결하면 커넥터가 m × n 개 필요. MCP는 가운데 레이어를 두어 각자 MCP에만 연결하면 되니까 m + n 으로 줄어듦. 즉 MCP는 **툴을 사용하는 표준 프로토콜**.

### 핵심 포인트
- 툴의 목적: 단순 계산/검색 자체가 아니라, **결과를 LLM에 컨텍스트로 넘겨 답변 품질을 높이는 것**.
- 사용 흐름: `bind_tools` → LLM이 `tool_calls` 반환 → 우리가 `tool.invoke(args)` → `ToolMessage`로 다시 LLM에 전달 → 최종 답변.
- 이번 주부터는 정답이 없음. 자유롭게 작성, 패들렛 다른 분 코드도 참고.

## 다룰 툴과 에이전트 패턴

### 오늘 다루는 내장 툴
| 툴 | 패키지 / import | 메모 |
| --- | --- | --- |
| Tavily 검색 | `langchain_community.tools.TavilySearchResults` | API 키 필요(`TAVILY_API_KEY`), 월 1000건 무료. `max_results=3` |
| Wikipedia | `WikipediaQueryRun` + `WikipediaAPIWrapper` | `top_k_results`, `doc_content_chars_max`, `lang="ko"` |
| DuckDuckGo | `DuckDuckGoSearchResults` + `DuckDuckGoSearchAPIWrapper` | API 키 불필요 |
| Python REPL | `langchain_experimental.tools.PythonREPLTool` | 별도 설치: `pip install langchain-experimental`. 어제 `eval` 기반 calculator의 확장판 — 파이썬 코드 자체를 실행 |

### 에이전트(Agent)란
"에이전트 에이전트" 하지만 결국 **LLM이 스스로 어떤 툴을 쓸지, 어떻게 인자를 넘길지, 결과를 어떻게 합칠지 판단하는 것**. 여러 툴을 `bind_tools([...])`로 한꺼번에 묶고 `tool_map = {t.name: t for t in tools}`로 이름→툴 매핑을 만들어 루프를 돈다.

### 강의 메모
- **무한 루프 주의**: `max_turns`(예: 3) 꼭 걸기. 안 그러면 토큰만 소모.
- **PythonREPL 한계**: LLM이 `print()` 없이 그냥 표현식만 만들면 출력이 비어버림. 프롬프트에 "출력해줘" 명시하거나 후처리 함수(`run_code_agent` 같은)로 포맷 통일.
- **검색 툴 비교**: 시간 측정해서 어떤 게 우리 LLM 컨텍스트로 좋은지 평가 가능 (RAG의 evaluation metric과 같은 맥락).
- **확장 방향**: Self-RAG / GraphRAG처럼 RAG 로직 + 툴 + 에이전트 조합으로 고도화 가능.